In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ШАГ 1. Получить точки-признаки из SIFT (блок SIFT+FLANN+RANSAC)

import cv2
import numpy as np
import json
import os
import time

DATA_ROOT = "/content/drive/MyDrive/VAC_HOLOD/test-task"
SPLIT_FILE = os.path.join(DATA_ROOT, "split.json")
OUTPUT_DIR = "solution4"
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(SPLIT_FILE, "r", encoding="utf-8") as f:
    split_data = json.load(f)
train_sessions = split_data.get("train", [])

sift_cache = {"top": {"src": [], "dst": []}, "bottom": {"src": [], "dst": []}}

def find_img_in_session(sess_dir, json_abs_path):
    fname = os.path.basename(json_abs_path)
    for sub in ["top", "bottom", "door2"]:
        p = os.path.join(sess_dir, sub, fname)
        if os.path.exists(p):
            return p
    return None

print("ШАГ 1: Извлечение и фильтрация SIFT...")
t0 = time.time()
debug_done = False

for sess_rel in train_sessions:
    sess_dir = os.path.join(DATA_ROOT, sess_rel)
    if not os.path.exists(sess_dir): continue

    for cam in ["top", "bottom"]:
        coords_path = os.path.join(sess_dir, f"coords_{cam}.json")
        if not os.path.exists(coords_path): continue

        with open(coords_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        for pair in data:
            if cam in pair["file1_path"]:
                path_json_src = pair["file1_path"]
                path_json_dst = pair["file2_path"]
            else:
                path_json_src = pair["file2_path"]
                path_json_dst = pair["file1_path"]

            img1_path = find_img_in_session(sess_dir, path_json_src)
            img2_path = find_img_in_session(sess_dir, path_json_dst)

            if not debug_done:
                print(f"Отладка путей ({cam}):")
                print(f"  JSON: {path_json_src}")
                print(f"  Найдено: {img1_path}")
                debug_done = True

            if img1_path is None or img2_path is None:
                continue

            img1 = cv2.imread(img1_path)
            img2 = cv2.imread(img2_path)
            if img1 is None or img2 is None:
                continue

            # Блок SIFT
            sift = cv2.SIFT_create(nfeatures=5000)
            kp1, des1 = sift.detectAndCompute(img1, None)
            kp2, des2 = sift.detectAndCompute(img2, None)
            if des1 is None or des2 is None: continue

            FLANN_INDEX_KDTREE = 1
            index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
            search_params = dict(checks=50)
            flann = cv2.FlannBasedMatcher(index_params, search_params)
            matches = flann.knnMatch(des1, des2, k=2)

            good_matches = []
            for m, n in matches:
                if m.distance < 0.75 * n.distance:
                    good_matches.append(m)

            if len(good_matches) >= 4:
                pts1 = np.array([kp1[m.queryIdx].pt for m in good_matches], dtype=np.float32)
                pts2 = np.array([kp2[m.trainIdx].pt for m in good_matches], dtype=np.float32)

                H, mask = cv2.findHomography(pts1, pts2, cv2.RANSAC, 3.0)
                if mask is not None:
                    mask = mask.ravel().astype(bool)
                    sift_cache[cam]["src"].extend(pts1[mask].tolist())
                    sift_cache[cam]["dst"].extend(pts2[mask].tolist())

    print(f"  Готово: {sess_rel}")

for cam in ["top", "bottom"]:
    src_arr = np.array(sift_cache[cam]["src"], dtype=np.float32)
    dst_arr = np.array(sift_cache[cam]["dst"], dtype=np.float32)
    np.savez(os.path.join(OUTPUT_DIR, f"sift_{cam}.npz"), src=src_arr, dst=dst_arr)
    print(f"SIFT {cam}: {len(src_arr)} чистых совпадений")

print(f"Время шага 1: {time.time()-t0:.1f} сек")

ШАГ 1: Извлечение и фильтрация SIFT...
Отладка путей (top):
  JSON: /tmp/labeling_duo_coord_uploads/bQYhkwltdKchjivcianTb3_jc_GSXVfe/5a75758cd32de846/folder2/top/frame_000159.jpg
  Найдено: /content/drive/MyDrive/VAC_HOLOD/test-task/train/camera_door2_2025-11-27_14-26-36/top/frame_000159.jpg
  Готово: train/camera_door2_2025-11-27_14-26-36
  Готово: train/camera_door2_2025-11-27_15-01-50
  Готово: train/camera_door2_2025-11-27_15-07-15
  Готово: train/camera_door2_2025-11-27_15-08-30
  Готово: train/camera_door2_2025-11-27_15-11-18
  Готово: train/camera_door2_2025-11-27_15-12-09
  Готово: train/camera_door2_2025-11-27_15-12-25
  Готово: train/camera_door2_2025-11-27_15-12-41
  Готово: train/camera_door2_2025-11-27_15-13-46
  Готово: train/camera_door2_2025-11-27_15-14-56
  Готово: train/camera_door2_2025-11-27_16-17-24
  Готово: train/camera_door2_2025-11-27_16-20-59
  Готово: train/camera_door2_2025-11-27_16-21-27
  Готово: train/camera_door2_2025-11-27_16-22-53
  Готово: train/camer

In [ ]:

# ШАГ 2. Слияние ручных точек + SIFT и расчёт метрики

'''
Загрузить ручные точки из train и объединить их с sift_*.npz.
Сохраняет единые top_ref.npz / bottom_ref.npz, прогоняет валидацию и пишет metrics.json.
'''

import cv2
import numpy as np
import json
import os

DATA_ROOT = "/content/drive/MyDrive/VAC_HOLOD/test-task"
SPLIT_FILE = os.path.join(DATA_ROOT, "split.json")
OUTPUT_DIR = "solution3"

with open(SPLIT_FILE, "r", encoding="utf-8") as f:
    split_data = json.load(f)

train_sessions = split_data.get("train", [])
val_sessions = split_data.get("val", [])

# Загрузка ручных точек из train
print("Загрузка ручных точек...")
manual_src = {"top": [], "bottom": []}
manual_dst = {"top": [], "bottom": []}

for sess_rel in train_sessions:
    sess_dir = os.path.join(DATA_ROOT, sess_rel)
    for cam in ["top", "bottom"]:
        coords_path = os.path.join(sess_dir, f"coords_{cam}.json")
        if not os.path.exists(coords_path): continue
        with open(coords_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        for pair in data:
            if cam in pair["file1_path"]:
                src_list = pair["image1_coordinates"]
                dst_list = pair["image2_coordinates"]
            else:
                src_list = pair["image2_coordinates"]
                dst_list = pair["image1_coordinates"]
            for s, d in zip(src_list, dst_list):
                manual_src[cam].append([s["x"], s["y"]])
                manual_dst[cam].append([d["x"], d["y"]])

# Загрузка SIFT точек и слияние
print("Слияние с SIFT...")
final_src, final_dst = {}, {}
for cam in ["top", "bottom"]:
    sift_path = os.path.join(OUTPUT_DIR, f"sift_{cam}.npz")
    sift_src_list, sift_dst_list = [], []
    if os.path.exists(sift_path):
        d = np.load(sift_path)
        sift_src_list = d["src"].tolist()
        sift_dst_list = d["dst"].tolist()

    combined_src = manual_src[cam] + sift_src_list
    combined_dst = manual_dst[cam] + sift_dst_list

    final_src[cam] = np.array(combined_src, dtype=np.float32)
    final_dst[cam] = np.array(combined_dst, dtype=np.float32)

    np.savez(os.path.join(OUTPUT_DIR, f"{cam}_ref.npz"), src=final_src[cam], dst=final_dst[cam])
    print(f"  {cam}_ref.npz: {len(combined_src)} точек (ручные + SIFT)")

# Функция предсказания (та же логика: K=5, thresh=1.0, weighted bias)
def predict(x, y, source, K=3):
    src_pts = final_src[source]
    dst_pts = final_dst[source]
    query = np.array([x, y], dtype=np.float32)

    dists = np.linalg.norm(src_pts - query, axis=1)
    idx = np.argsort(dists)[:K]
    p_src, p_dst = src_pts[idx], dst_pts[idx]

    try:
        H, status = cv2.findHomography(p_src, p_dst, method=cv2.RANSAC, ransacReprojThreshold=1.0)
        if H is None or status is None: raise ValueError
        mask = status.ravel().astype(bool)
        if mask.sum() < 4: raise ValueError

        H, _ = cv2.findHomography(p_src[mask], p_dst[mask], cv2.RANSAC)
        mapped = cv2.perspectiveTransform(p_src[mask].reshape(-1, 1, 2), H)[:, 0]

        dists_local = np.linalg.norm(p_src[mask] - query, axis=1) + 1e-6
        weights = 1.0 / dists_local
        weights /= weights.sum()
        bias = np.sum(weights[:, None] * (p_dst[mask] - mapped), axis=0)

        pred = cv2.perspectiveTransform(np.array([[[x, y]]], dtype=np.float32), H)[0][0]
        return float(pred[0] + bias[0]), float(pred[1] + bias[1])
    except Exception:
        fb = query + np.mean(p_dst - p_src, axis=0)
        return float(fb[0]), float(fb[1])

# Валидация на val сплите
print("\nРасчёт метрики на val...")
errors_top, errors_bottom = [], []

for sess_rel in val_sessions:
    sess_dir = os.path.join(DATA_ROOT, sess_rel)
    for cam in ["top", "bottom"]:
        coords_path = os.path.join(sess_dir, f"coords_{cam}.json")
        if not os.path.exists(coords_path): continue
        with open(coords_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        for pair in data:
            if cam in pair["file1_path"]:
                src_list = pair["image1_coordinates"]
                dst_list = pair["image2_coordinates"]
            else:
                src_list = pair["image2_coordinates"]
                dst_list = pair["image1_coordinates"]
            for s, d in zip(src_list, dst_list):
                px, py = predict(s["x"], s["y"], cam)
                err = np.hypot(px - d["x"], py - d["y"])
                if cam == "top": errors_top.append(err)
                else: errors_bottom.append(err)

med_top = float(np.mean(errors_top)) if errors_top else 0.0
med_bottom = float(np.mean(errors_bottom)) if errors_bottom else 0.0

metrics = {"top_to_door2_med": med_top, "bottom_to_door2_med": med_bottom, "overall_med": (med_top+med_bottom)/2}
with open(os.path.join(OUTPUT_DIR, "metrics.json"), "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print(f"MED top -> door2:    {med_top:.2f} px")
print(f"MED bottom -> door2: {med_bottom:.2f} px")
print("metrics.json сохранён")

Загрузка ручных точек...
Слияние с SIFT...
  top_ref.npz: 3835 точек (ручные + SIFT)
  bottom_ref.npz: 3369 точек (ручные + SIFT)

Расчёт метрики на val...
MED top -> door2:    140.41 px
MED bottom -> door2: 154.75 px
metrics.json сохранён
